<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/model-risk/lessons/P04-L10-committee-pack/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/model-risk/lessons/P04-L10-committee-pack/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/model-risk/lessons/P04-L10-committee-pack/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/model-risk/lessons/P04-L10-committee-pack/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P04-L10 · The validation report and the committee pack

**You will build:** the document a model risk committee actually receives — a findings
register with owners, due dates and statuses; a remediation tracker that ages what is still
open and escalates what is late; an executive summary assembled from the findings rather than
written above them; a limitations section built from what the suite could not measure — and
then the gate that makes the pack prove itself. Every figure in it is traced to a computed
result, or the pack refuses to render.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** T00-L01 (the tier gate and the profiler), P04-L01 (the validation suite
and its report, carried forward here), and modules 2 to 9 of this programme, whose records
this pack assembles. Pure numpy and the standard library.

The data is **synthetic and generated in this notebook**, and the evidence from the other
modules arrives as small fixtures in each module's own record format. Every figure you see
is computed by code you run — except one, and finding it is the point.

By the end you will be able to:

1. Implement a findings register whose identifiers and raised dates survive from one pack to
   the next, so a finding that recurs keeps its age.
2. Implement a remediation tracker that ages live findings from an explicit as-of date and
   escalates the overdue ones by policy.
3. Assemble an executive summary and a limitations section from the evidence, not above it.
4. Define precisely what counts as a figure in a document, and implement a gate that traces
   every one of them to a computed result at the precision it was printed to.
5. Explain why a gate that widens its tolerance, skips a class of numbers or reads numbers
   out of strings passes the very document it exists to stop.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import datetime as _dt
import hashlib
import io
import math
import re
import sys
import time
import traceback
from typing import Any, Callable, Mapping, NamedTuple

import numpy as np

SEED = 20260916          # module 1's seed: the development and monitoring samples ARE module 1's
N_DEV = 6000
N_MON = 6000
N_OOT = 1400             # module 4's out-of-time size: a later and much smaller period
OOT_DRIFT = -0.6         # the later period is lower-risk: the credit policy was tightened
N_BINS = 10
MIN_SUPPORT = 200
PSI_THRESHOLD = 0.25
PSI_FLOOR = 1e-6
POLICY = {"min_auc": 0.65, "max_ece": 0.04, "min_auc_gain": 0.02}   # module 1's promotion rule
N_BOOT = 400             # bootstrap replicates behind every interval here
ALPHA = 0.05
POWER = 0.80
MATERIAL_AUC_DIFF = 0.010
AS_OF = "2026-09-30"     # this pack's date. Every age is measured from it, never from the clock
MODEL_ID = "champion-v3 (retail application scorecard)"

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__)

DATA_NOTE = (
    "SYNTHETIC DATA. Every record below was generated inside this notebook by "
    f"numpy.random.default_rng({SEED}). No real applicant, account or lending decision is "
    "represented. The evidence from the other modules of this programme is carried as small "
    "fixtures in each module's own record format; a pack on real data takes those records "
    "from the runs that produced them, and names the extract, its date and its lineage here."
)

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("build_register",),
    "exercise 2": ("age_register",),
    "exercise 3": ("executive_summary",),
    "exercise 4": ("limitations",),
    "exercise 5": ("extract_figures",),
    "exercise 6": ("computed_values",),
    "exercise 7": ("trace_gate",),
    "exercise 8": ("render_committee_pack",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (executive_summary)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def parse_date(text: str) -> _dt.date:
    """Parse a `YYYY-MM-DD` string. Module 2's helper, carried forward; raises ValueError on
    anything else, so a malformed date never becomes a silently wrong age."""
    if not isinstance(text, str) or not re.fullmatch(r"\d{4}-\d{2}-\d{2}", text):
        raise ValueError(f"not a YYYY-MM-DD date: {text!r}")
    return _dt.date.fromisoformat(text)


print("\n" + DATA_NOTE)

## 1. Module 1's run, carried forward

Module 1 built six functions and a report generator. This notebook cannot import that lesson
— it has to open on its own, anywhere — so the cell below carries a minimal faithful copy of
each, under the same names, and runs the same suite on the same seed. The development and
monitoring samples are module 1's, record for record. One out-of-time sample is added: a
later, smaller period, drawn from the same generator after the credit policy tightened.

Nothing here is an exercise. Read it for the record formats: `ReliabilityTable`,
`StabilityResult`, the subgroup table and `Decision` all reach the committee pack unchanged.

In [ ]:
# --- module 1, carried forward (P04-L01). Given; not graded. -------------------------------
def _sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def synthetic_portfolio(rng: np.random.Generator, n: int, drift: float = 0.0) -> dict:
    """Module 1's generator, unchanged. SYNTHETIC — see DATA_NOTE."""
    segment_names = np.array(["A", "B", "C", "D", "E"])
    seg_idx = rng.choice(5, size=n, p=[0.34, 0.28, 0.22, 0.14, 0.02])
    seg_offset = np.array([0.0, -0.25, 0.30, 0.55, -0.10])[seg_idx]
    z = rng.normal(0.0, 1.0, n) + drift + seg_offset
    true_logit = -1.55 + 0.95 * z
    y = (rng.random(n) < _sigmoid(true_logit)).astype(np.int64)
    champ_logit = 1.30 * (-1.55 + 0.95 * (z + rng.normal(0.0, 0.62, n))) + 0.18
    champ_logit = champ_logit - 0.55 * (seg_idx == 3)
    chal_logit = -1.55 + 0.95 * (z + rng.normal(0.0, 0.26, n))
    return {"segment": segment_names[seg_idx], "y": y,
            "champion": _sigmoid(champ_logit), "challenger": _sigmoid(chal_logit)}


def quantile_edges(x: np.ndarray, n_bins: int = N_BINS) -> np.ndarray:
    """Module 1's stability edges: cut once on the baseline, outer edges opened to infinity."""
    edges = np.unique(np.quantile(np.asarray(x, dtype=float), np.linspace(0.0, 1.0, n_bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    return edges


class ReliabilityTable(NamedTuple):
    lo: np.ndarray          # lower edge of each bin
    hi: np.ndarray          # upper edge of each bin
    count: np.ndarray       # records in the bin; 0 for an empty bin, which keeps its row
    mean_pred: np.ndarray   # nan when empty
    obs_rate: np.ndarray    # nan when empty


def reliability_table(y_true: np.ndarray, y_prob: np.ndarray,
                      n_bins: int = N_BINS) -> ReliabilityTable:
    y = np.asarray(y_true)
    p = np.asarray(y_prob, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.searchsorted(edges, p, side="left") - 1, 0, n_bins - 1)
    count = np.bincount(idx, minlength=n_bins).astype(np.int64)
    pred_sum = np.bincount(idx, weights=p, minlength=n_bins)
    event_sum = np.bincount(idx, weights=y.astype(float), minlength=n_bins)
    with np.errstate(invalid="ignore", divide="ignore"):
        denom = np.where(count > 0, count, np.nan)
        return ReliabilityTable(edges[:-1], edges[1:], count, pred_sum / denom, event_sum / denom)


def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray,
                               n_bins: int = N_BINS) -> float:
    table = reliability_table(y_true, y_prob, n_bins)
    filled = table.count > 0
    weights = table.count[filled] / int(table.count.sum())
    return float(np.sum(weights * np.abs(table.obs_rate[filled] - table.mean_pred[filled])))


class StabilityResult(NamedTuple):
    psi: float
    contributions: np.ndarray    # per-bin (a - e) * ln(a / e), summing to psi
    expected_pct: np.ndarray
    actual_pct: np.ndarray


def population_stability_index(expected: np.ndarray, actual: np.ndarray, edges: np.ndarray,
                               floor: float = PSI_FLOOR) -> StabilityResult:
    edges = np.asarray(edges, dtype=float)
    n_bins = edges.size - 1

    def _shares(x: np.ndarray) -> np.ndarray:
        v = np.asarray(x, dtype=float)
        idx = np.clip(np.searchsorted(edges, v, side="right") - 1, 0, n_bins - 1)
        return np.bincount(idx, minlength=n_bins).astype(float) / v.size

    e_pct, a_pct = _shares(expected), _shares(actual)
    e_safe, a_safe = np.maximum(e_pct, floor), np.maximum(a_pct, floor)
    contributions = (a_safe - e_safe) * np.log(a_safe / e_safe)
    return StabilityResult(float(contributions.sum()), contributions, e_pct, a_pct)


def subgroup_table(y_true: np.ndarray, y_prob: np.ndarray, groups: np.ndarray,
                   min_support: int = MIN_SUPPORT, n_bins: int = N_BINS) -> dict:
    y, p, g = np.asarray(y_true), np.asarray(y_prob, dtype=float), np.asarray(groups)
    names = np.unique(g)
    count = np.zeros(names.size, dtype=np.int64)
    event_rate, mean_pred, ece = (np.full(names.size, np.nan) for _ in range(3))
    reportable = np.zeros(names.size, dtype=bool)
    for i, name in enumerate(names):
        mask = g == name
        count[i] = int(mask.sum())
        reportable[i] = count[i] >= min_support
        if reportable[i]:
            event_rate[i] = float(y[mask].mean())
            mean_pred[i] = float(p[mask].mean())
            ece[i] = expected_calibration_error(y[mask], p[mask], n_bins)
    return {"group": names, "count": count, "event_rate": event_rate, "mean_pred": mean_pred,
            "gap": mean_pred - event_rate, "ece": ece, "reportable": reportable}


class Decision(NamedTuple):
    verdict: str        # "promote" | "hold" | "reject"
    reasons: tuple      # one "PASS ..."/"FAIL ..." string per rule, in rule order
    margins: dict


def challenger_decision(champion: Mapping[str, float], challenger: Mapping[str, float],
                        policy: Mapping[str, float]) -> Decision:
    gain = float(challenger["auc"]) - float(champion["auc"])
    ece_change = float(challenger["ece"]) - float(champion["ece"])
    tests = (
        (float(challenger["auc"]) >= float(policy["min_auc"]),
         f"challenger AUC {challenger['auc']:.4f} against the floor of {policy['min_auc']:.4f}"),
        (float(challenger["ece"]) <= float(policy["max_ece"]),
         f"challenger ECE {challenger['ece']:.4f} against the ceiling of "
         f"{policy['max_ece']:.4f}"),
        (gain >= float(policy["min_auc_gain"]),
         f"AUC gain {gain:+.4f} against the required {policy['min_auc_gain']:.4f}"),
        (ece_change <= 0.0, f"calibration change {ece_change:+.4f} against a ceiling of +0.0000"),
    )
    reasons = tuple(("PASS " if ok else "FAIL ") + text for ok, text in tests)
    if not (tests[0][0] and tests[1][0]):
        verdict = "reject"
    elif tests[2][0] and tests[3][0]:
        verdict = "promote"
    else:
        verdict = "hold"
    margins = {"auc_gain": gain, "ece_change": ece_change,
               "auc_headroom": float(challenger["auc"]) - float(policy["min_auc"]),
               "ece_headroom": float(policy["max_ece"]) - float(challenger["ece"])}
    return Decision(verdict, reasons, margins)


REPORT_SECTIONS = ("## 1. Data", "## 2. Calibration", "## 3. Population stability",
                   "## 4. Subgroup performance", "## 5. Champion versus challenger",
                   "## 6. Decision")


def render_validation_report(findings: Mapping[str, Any]) -> str:
    """Module 1's report generator, unchanged: every figure formatted out of `findings`."""
    required = ("model_id", "as_of", "data_note", "reliability", "ece", "psi", "psi_threshold",
                "subgroups", "min_support", "champion", "challenger", "decision")
    missing = [k for k in required if k not in findings]
    if missing:
        raise ValueError("findings is missing: " + ", ".join(missing))
    rel, psi, sub, dec = (findings["reliability"], findings["psi"], findings["subgroups"],
                          findings["decision"])
    thr = float(findings["psi_threshold"])
    out = [f"# Validation report — {findings['model_id']}", "",
           f"As of {findings['as_of']}. Generated by the validation suite from the run that "
           "produced the figures below; no number in this document was typed by hand.", "",
           REPORT_SECTIONS[0], "", str(findings["data_note"]), "",
           REPORT_SECTIONS[1], "",
           f"Expected calibration error: {float(findings['ece']):.4f}", "",
           "| bin | mean predicted | observed rate | n |", "| --- | --- | --- | --- |"]
    for lo, hi, mean_pred, obs_rate, count in zip(rel.lo, rel.hi, rel.mean_pred,
                                                  rel.obs_rate, rel.count):
        if int(count) == 0:
            out.append(f"| {lo:.2f}-{hi:.2f} | n/a | n/a | 0 |")
        else:
            out.append(f"| {lo:.2f}-{hi:.2f} | {mean_pred:.4f} | {obs_rate:.4f} | {int(count)} |")
    worst = int(np.argmax(psi.contributions))
    out += ["", REPORT_SECTIONS[2], "",
            f"Population stability index: {float(psi.psi):.4f} against a policy threshold of "
            f"{thr:.2f}.",
            f"Largest single-bin contribution: bin {worst} at "
            f"{float(psi.contributions[worst]):.4f} "
            f"({float(psi.expected_pct[worst]):.4f} of the baseline against "
            f"{float(psi.actual_pct[worst]):.4f} now).",
            f"Verdict: {'BREACH' if float(psi.psi) > thr else 'WITHIN THRESHOLD'}",
            "", REPORT_SECTIONS[3], "",
            f"Minimum support for a reported metric: {int(findings['min_support'])} records.",
            "", "| group | n | mean predicted | observed rate | gap | ECE |",
            "| --- | --- | --- | --- | --- | --- |"]
    for i, name in enumerate(sub["group"]):
        n = int(sub["count"][i])
        if not bool(sub["reportable"][i]):
            out.append(f"| {name} | {n} | suppressed (n={n} < minimum support "
                       f"{int(findings['min_support'])}) | | | |")
        else:
            out.append(f"| {name} | {n} | {float(sub['mean_pred'][i]):.4f} | "
                       f"{float(sub['event_rate'][i]):.4f} | {float(sub['gap'][i]):+.4f} | "
                       f"{float(sub['ece'][i]):.4f} |")
    champ, chal = findings["champion"], findings["challenger"]
    out += ["", REPORT_SECTIONS[4], "", "| metric | champion | challenger |",
            "| --- | --- | --- |",
            f"| AUC | {float(champ['auc']):.3f} | {float(chal['auc']):.3f} |",
            f"| ECE | {float(champ['ece']):.4f} | {float(chal['ece']):.4f} |",
            "", REPORT_SECTIONS[5], "", f"Verdict: {str(dec.verdict).upper()}", ""]
    out += [f"- {reason}" for reason in dec.reasons]
    out += ["", "Margins: " + ", ".join(f"{k}={v:+.4f}" for k, v in dec.margins.items()), ""]
    return "\n".join(out)


# --- module 4, carried forward (P04-L04): the rank AUC, its intervals, and measurability ----
def average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    _, inverse, counts = np.unique(x[order], return_inverse=True, return_counts=True)
    block_start = np.concatenate(([0], np.cumsum(counts)[:-1]))
    out = np.empty(x.size, dtype=float)
    out[order] = block_start[inverse] + (counts[inverse] - 1) / 2.0 + 1.0
    return out


def auc_by_ranks(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y = np.asarray(y_true)
    n_pos = int((y == 1).sum())
    n_neg = int(y.size - n_pos)
    ranks = average_ranks(y_score)
    return float((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def normal_cdf(z: float) -> float:
    return 0.5 * math.erfc(-float(z) / math.sqrt(2.0))


def normal_quantile(q: float) -> float:
    lo, hi = -40.0, 40.0
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if normal_cdf(mid) < q:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


class Interval(NamedTuple):
    point: float
    lo: float
    hi: float
    se: float
    n_boot: int


def bootstrap_auc_ci(y_true: np.ndarray, y_score: np.ndarray, n_boot: int = N_BOOT,
                     alpha: float = ALPHA, rng: np.random.Generator | None = None) -> Interval:
    y, p = np.asarray(y_true), np.asarray(y_score, dtype=float)
    rng = np.random.default_rng(SEED) if rng is None else rng
    idx_pos, idx_neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    reps = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        take = np.concatenate([rng.choice(idx_pos, idx_pos.size, replace=True),
                               rng.choice(idx_neg, idx_neg.size, replace=True)])
        reps[b] = auc_by_ranks(y[take], p[take])
    lo, hi = np.quantile(reps, [alpha / 2.0, 1.0 - alpha / 2.0])
    return Interval(auc_by_ranks(y, p), float(lo), float(hi), float(reps.std(ddof=1)),
                    int(n_boot))


class PairedDifference(NamedTuple):
    diff: float             # auc_a - auc_b on the original sample
    lo: float
    hi: float
    se_paired: float
    se_independent: float
    corr: float
    n_boot: int


def paired_bootstrap_difference(y_true: np.ndarray, score_a: np.ndarray, score_b: np.ndarray,
                                n_boot: int = N_BOOT, alpha: float = ALPHA,
                                rng: np.random.Generator | None = None) -> PairedDifference:
    y = np.asarray(y_true)
    a, b = np.asarray(score_a, dtype=float), np.asarray(score_b, dtype=float)
    rng = np.random.default_rng(SEED) if rng is None else rng
    idx_pos, idx_neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    reps_a, reps_b = np.empty(n_boot), np.empty(n_boot)
    for i in range(n_boot):
        take = np.concatenate([rng.choice(idx_pos, idx_pos.size, replace=True),
                               rng.choice(idx_neg, idx_neg.size, replace=True)])
        reps_a[i] = auc_by_ranks(y[take], a[take])
        reps_b[i] = auc_by_ranks(y[take], b[take])
    reps_d = reps_a - reps_b
    lo, hi = np.quantile(reps_d, [alpha / 2.0, 1.0 - alpha / 2.0])
    se_a, se_b = float(reps_a.std(ddof=1)), float(reps_b.std(ddof=1))
    return PairedDifference(auc_by_ranks(y, a) - auc_by_ranks(y, b), float(lo), float(hi),
                            float(reps_d.std(ddof=1)), float(math.hypot(se_a, se_b)),
                            float(np.corrcoef(reps_a, reps_b)[0, 1]), int(n_boot))


class Measurability(NamedTuple):
    mdd: float          # minimum detectable difference at (alpha, power)
    n_required: int     # records needed before `material` becomes detectable
    verdict: str        # SIGNIFICANT | NOT SIGNIFICANT | NOT MEASURABLE
    reason: str


def measurability(observed_diff: float, se: float, n: int, material: float = MATERIAL_AUC_DIFF,
                  alpha: float = ALPHA, power: float = POWER) -> Measurability:
    z_crit = normal_quantile(1.0 - alpha / 2.0)
    mdd = (z_crit + normal_quantile(power)) * se
    n_required = int(math.ceil(n * (mdd / material) ** 2))
    head = (f"mdd={mdd:.4f} material={material:.4f} crit={z_crit * se:.4f} "
            f"observed={abs(observed_diff):.4f}")
    if abs(observed_diff) >= z_crit * se:
        verdict, tail = "SIGNIFICANT", "the observed difference clears the critical value"
    elif mdd > material:
        verdict, tail = "NOT MEASURABLE", (f"{n_required} records would be needed before the "
                                           "comparison could conclude anything")
    else:
        verdict, tail = "NOT SIGNIFICANT", "the sample could have seen a material difference"
    return Measurability(float(mdd), n_required, verdict, f"{head}; {tail}")


# --- the run --------------------------------------------------------------------------------
_rng = np.random.default_rng(SEED)
DEV = synthetic_portfolio(_rng, N_DEV, drift=0.0)          # module 1's development sample
MON = synthetic_portfolio(_rng, N_MON, drift=1.00)         # module 1's monitoring sample
OOT = synthetic_portfolio(_rng, N_OOT, drift=OOT_DRIFT)    # added here: a later, smaller period


def run_suite() -> dict:
    """Module 1's suite on module 1's data, plus the few inputs its report prints unregistered."""
    edges = quantile_edges(DEV["champion"], N_BINS)
    champion = {"auc": auc_by_ranks(DEV["y"], DEV["champion"]),
                "ece": expected_calibration_error(DEV["y"], DEV["champion"], N_BINS)}
    challenger = {"auc": auc_by_ranks(DEV["y"], DEV["challenger"]),
                  "ece": expected_calibration_error(DEV["y"], DEV["challenger"], N_BINS)}
    psi = population_stability_index(DEV["champion"], MON["champion"], edges)
    return {
        "model_id": MODEL_ID, "as_of": AS_OF, "data_note": DATA_NOTE,
        "reliability": reliability_table(DEV["y"], DEV["champion"], N_BINS),
        "ece": champion["ece"], "psi": psi, "psi_threshold": PSI_THRESHOLD,
        "subgroups": subgroup_table(DEV["y"], DEV["champion"], DEV["segment"], MIN_SUPPORT),
        "min_support": MIN_SUPPORT, "champion": champion, "challenger": challenger,
        "decision": challenger_decision(champion, challenger, POLICY),
        # Inputs the report prints without module 1 ever registering them: the seed inside
        # the data note, the size and event rate of the sample, the promotion policy, and the
        # one figure module 1's renderer derives for itself — the bin it names as the worst.
        "seed": SEED, "n_records": N_DEV, "event_rate": float(DEV["y"].mean()),
        "policy": dict(POLICY), "psi_worst_bin": int(np.argmax(psi.contributions)),
    }


VALIDATION = run_suite()
print(f"module 1's suite, re-run: champion ECE {VALIDATION['ece']:.4f}, "
      f"PSI {VALIDATION['psi'].psi:.4f}, challenger verdict {VALIDATION['decision'].verdict}")
print(f"out-of-time sample: {N_OOT} records, event rate {OOT['y'].mean():.4f}")

## 2. The pack you are handed

The model owner has assembled this quarter's committee pack the way packs are usually
assembled: module 1's generated report at the bottom, and an executive summary typed above
it. The helper below builds that document. It formats every figure in the summary out of
the run, standing in for a person who copied them correctly — except one, which it writes
exactly as a person typed it.

Read the summary. Every number in it looks like the others. The committee will read this
page and nothing below it, and one of those numbers is not a result of anything.

In [ ]:
def draft_pack(validation: Mapping[str, Any]) -> str:
    """The pack as the model owner assembled it: a summary typed above module 1's report. GIVEN."""
    v = validation
    summary = [
        f"# Committee pack — {v['model_id']}", "",
        f"As of {v['as_of']}. Prepared by the model owner.", "",
        "## 1. Executive summary", "",
        "Opinion: fit for use with conditions.", "",
        f"The champion was validated on {v['n_records']:,} development records with an "
        f"observed event rate of {v['event_rate']:.1%}. Its expected calibration error of "
        f"{v['ece']:.4f} exceeds the policy ceiling of {v['policy']['max_ece']:.2f}, and "
        "findings F-001 and F-006 carried from the last pack remain open. The population "
        f"stability index of {v['psi'].psi:.4f} breaches its threshold of "
        f"{v['psi_threshold']:.2f}.", "",
        "The challenger improves calibration to 0.0140 and is recommended for promotion.", "",
        "## 2. Validation report", "",
    ]
    report = render_validation_report(v).splitlines()[1:]
    return "\n".join(summary + [("#" + line) if line.startswith("#") else line
                                for line in report])


DRAFT = draft_pack(VALIDATION)
print("\n".join(DRAFT.splitlines()[:14]))
print(f"\n... and {len(DRAFT.splitlines()) - 14} more lines of module 1's report below it.")

## 3. The evidence from the other modules

A committee pack is not one module's report. It gathers what every part of the suite found.
Each module of this programme returns its evidence in its own record format, and the cell
below builds one record of each kind — module 3's `Finding` and `MonotonicityProbe`, module
2's `CoverageRow`, module 4's `Interval` and `Measurability`, module 5's
`CharacteristicShift`, module 6's verdict, module 7's `Audit`, module 8's certificate and
module 9's monitor report — with the same names and fields those modules use.

Module 4's records are computed here, on the samples above, with module 4's own functions.
Module 5's are computed on two stand-in driver samples. The others are FIXTURES: the record
each module's run would return, written out, because running a C build or a fresh-process
regeneration inside this notebook would teach nothing new. The fixtures are labelled as such.

In [ ]:
# --- module 4 (P04-L04), computed on the samples above ---------------------------------------
_boot = np.random.default_rng(SEED + 4)
_intervals = {"development": bootstrap_auc_ci(DEV["y"], DEV["champion"], rng=_boot)}
for _seg in ("A", "B", "C", "D"):          # segment E is below minimum support: no interval
    _m = DEV["segment"] == _seg
    _intervals[f"segment {_seg}, development"] = bootstrap_auc_ci(DEV["y"][_m],
                                                                  DEV["champion"][_m], rng=_boot)
_paired, _comparisons = {}, {}
for _name, _mask in (("challenger vs champion, out-of-time", np.ones(N_OOT, dtype=bool)),
                     ("challenger vs champion, segment D, out-of-time", OOT["segment"] == "D")):
    _p = paired_bootstrap_difference(OOT["y"][_mask], OOT["challenger"][_mask],
                                     OOT["champion"][_mask], rng=_boot)
    _paired[_name] = _p
    _comparisons[_name] = measurability(_p.diff, _p.se_paired, int(_mask.sum()))


# --- module 5 (P04-L05): per-driver stability, on two stand-in driver samples ----------------
class CharacteristicShift(NamedTuple):
    driver: str
    csi: float                   # module 1's PSI on this driver, edges cut on its baseline
    bins: int
    log_odds_shift: float        # coefficient * (current mean - baseline mean)
    stability: StabilityResult


_drv = np.random.default_rng(SEED + 5)
_MODEL_LOG_ODDS = {"months_on_book": -0.0040, "utilisation": 3.5}     # module 5's model
_driver_samples = {
    "months_on_book": (_drv.normal(60.0, 18.0, N_DEV), _drv.normal(61.0, 18.0, N_MON)),
    "utilisation": (_drv.beta(2.0, 5.0, N_DEV), _drv.beta(3.0, 4.6, N_MON)),
}
_shifts = []
for _driver, (_base, _now) in _driver_samples.items():
    _edges = quantile_edges(_base, N_BINS)
    _st = population_stability_index(_base, _now, _edges)
    _shift = _MODEL_LOG_ODDS[_driver] * float(_now.mean() - _base.mean())
    _shifts.append(CharacteristicShift(_driver, _st.psi, int(_edges.size - 1), _shift, _st))


# --- module 2 (P04-L02), FIXTURE ------------------------------------------------------------
class TierDecision(NamedTuple):
    tier: int
    score: int
    reasons: tuple


class CoverageRow(NamedTuple):
    model_id: str
    months_since: int | None
    due_months: int
    verdict: str            # "never_validated" | "overdue" | "current"


# --- module 3 (P04-L03), FIXTURE ------------------------------------------------------------
class MonotonicityProbe(NamedTuple):
    variable: str
    direction: str
    violated: bool
    worst_violation: float
    counterexample: tuple
    n_skipped: int


class Finding(NamedTuple):
    rule: str        # the family: MONO, VAR-UNUSED, ...
    subject: str     # the variable or feature it concerns; "" for register-level findings
    severity: str
    detail: str


_mono = MonotonicityProbe("months_on_book", "decreasing", True, 0.012345,
                          (24.0, 36.0, 0.1012, 0.113545), 0)


# --- module 6 (P04-L06): the reproduction verdict, in Python, on FIXTURE totals --------------
def verdict_of(a: float, b: float, n: int, sum_abs: float) -> dict:
    """Module 6's rule: REPRODUCED if equal, EXPLAINED within (n - 1) * 2**-53 * sum_abs,
    FINDING otherwise. The same dict shape module 6's wrapper returns."""
    gap = abs(a - b)
    bound = (n - 1) * 2.0 ** -53 * abs(sum_abs)
    code = 0 if a == b else 1 if gap <= bound else 2
    return {"verdict": float(code), "gap": gap, "bound": bound,
            "name": ("REPRODUCED", "EXPLAINED", "FINDING")[code]}


_first_line_total = 5_734_912.25          # FIXTURE: the first line's portfolio aggregate
_dropped_position = 0.0068359375          # FIXTURE: the one exposure the second line's run lost
_sum_abs = _first_line_total              # FIXTURE: a lending book's exposures are all positive


# --- module 7 (P04-L07), FIXTURE ------------------------------------------------------------
class Check(NamedTuple):
    name: str        # "records" | "period" | "missing values"
    held: bool
    evidence: str


class Audit(NamedTuple):
    comparable: bool
    checks: tuple


def _held(name: str) -> Check:
    return Check(name, True, "every shared record agrees")


EVIDENCE: dict[str, Any] = {
    "calibration": {
        "ece": VALIDATION["ece"], "max_ece": POLICY["max_ece"],
        "reliability": {"champion, development": VALIDATION["reliability"],
                        "challenger, out-of-time": reliability_table(OOT["y"], OOT["challenger"])},
        "subgroups": VALIDATION["subgroups"], "min_support": MIN_SUPPORT,
    },
    "stability": {"psi": VALIDATION["psi"], "threshold": PSI_THRESHOLD,
                  "worst_bin": VALIDATION["psi_worst_bin"], "shifts": tuple(_shifts)},
    "tiering": {
        "tier": TierDecision(1, 9, ("materiality=high +3", "complexity=medium +2",
                                    "reversibility=irreversible +2", "score 9 -> tier 1")),
        "coverage": CoverageRow("champion-v3", 14, 12, "overdue"),
    },
    "soundness": {
        "probes": {"months_on_book": _mono},
        "findings": (Finding("MONO", "months_on_book", "high",
                             f"months_on_book is documented as {_mono.direction} and the sweep "
                             f"moved the wrong way by {_mono.worst_violation:.6f}"),),
    },
    "discrimination": {"intervals": _intervals, "min_auc": POLICY["min_auc"],
                       "paired": _paired, "comparisons": _comparisons,
                       "material": MATERIAL_AUC_DIFF, "alpha": ALPHA, "power": POWER},
    "reproduction": {
        "n": 2_000_000, "sum_abs": _sum_abs, "first_line_total": _first_line_total,
        "second_line_total": _first_line_total - _dropped_position,
        "verdict": verdict_of(_first_line_total, _first_line_total - _dropped_position,
                              2_000_000, _sum_abs),
    },
    "comparability": {"audits": {
        "challenger": Audit(True, (_held("records"), _held("period"), _held("missing values"))),
        "vendor-challenger": Audit(False, (
            _held("records"), _held("period"),
            Check("missing values", False, "income: missing in the reference and imputed with "
                                           "one repeated value in the vendor's frame"))),
    }},
    "explainability": {"certificate": {
        "reproduced": False, "seed": 20260923,
        "sha256": hashlib.sha256(b"fixture: the explanation pack in hand").hexdigest(),
        "n_bytes": 18433, "hash_seeds": [1, 2, 3],
        "runs": [{"hash_seed": 1, "match": True}, {"hash_seed": 2, "match": False},
                 {"hash_seed": 3, "match": True}],
        "first_difference": {"hash_seed": 2, "line": 41,
                             "expected": "  \"enquiries\": { ...", "got": "  \"income\": { ..."},
    }},
    "monitoring": {"report": {
        "extract_to": "2026-08-31", "records": 4_194_304, "threshold": PSI_THRESHOLD,
        "expected_counts": (20_000,) * 10,
        "counts": (280_000, 283_000, 286_000, 288_000, 289_000, 291_000, 293_000, 296_867,
                   1_048_576, 838_861),
    }},
}


def psi_from_counts_reference(expected_counts, actual_counts,
                              floor: float = PSI_FLOOR) -> StabilityResult:
    """Module 9's helper, carried forward: module 1's floored PSI, applied to bin COUNTS."""
    e = np.asarray(expected_counts, dtype=np.int64)
    a = np.asarray(actual_counts, dtype=np.int64)
    e_pct, a_pct = e.astype(float) / int(e.sum()), a.astype(float) / int(a.sum())
    e_safe, a_safe = np.maximum(e_pct, floor), np.maximum(a_pct, floor)
    contributions = (a_safe - e_safe) * np.log(a_safe / e_safe)
    return StabilityResult(float(contributions.sum()), contributions, e_pct, a_pct)


# Module 9's monitor reports its PSI from the bin counts, with module 1's floored arithmetic.
_mon = EVIDENCE["monitoring"]["report"]
assert sum(_mon["counts"]) == _mon["records"], "the fixture's counts must cover every record"
_mon_psi = psi_from_counts_reference(_mon["expected_counts"], _mon["counts"])
_mon.update(psi=_mon_psi.psi, contributions=_mon_psi.contributions,
            breach=bool(_mon_psi.psi > _mon["threshold"]),
            breach_bin=int(np.argmax(_mon_psi.contributions)))

Each record now has to become a finding, in module 3's `Finding` format, with a severity
from a written policy. `raise_findings()` below does that — one rule per kind of record, and
nothing raised for anything that passed. It is given: the rules are the policy's business,
not this lesson's, and module 3 already made you build a finding register once. Read it for
one thing: every figure a finding's `detail` carries is formatted from a NUMBER in the
evidence. Remember that when you reach exercise 6.

In [ ]:
SEVERITY_ORDER = ("critical", "high", "medium", "low")      # module 3's scale, most severe first
SEVERITY_CODES = {"critical": "S1", "high": "S2", "medium": "S3", "low": "S4"}
SOURCES = ("calibration", "stability", "tiering", "soundness", "discrimination",
           "reproduction", "comparability", "explainability", "monitoring")
MODULE_OF = {"calibration": "P04-L01", "stability": "P04-L01, P04-L05", "tiering": "P04-L02",
             "soundness": "P04-L03", "discrimination": "P04-L04", "reproduction": "P04-L06",
             "comparability": "P04-L07", "explainability": "P04-L08", "monitoring": "P04-L09"}

PACK_POLICY: dict[str, Any] = {
    "owners": {"calibration": "retail credit modelling", "stability": "retail credit modelling",
               "tiering": "model risk inventory", "soundness": "retail credit modelling",
               "discrimination": "retail credit modelling",
               "reproduction": "credit risk reporting", "comparability": "vendor model oversight",
               "explainability": "model explainability", "monitoring": "model monitoring"},
    "days_to_remediate": {"critical": 30, "high": 90, "medium": 180, "low": 365},
    # (days overdue at which it applies, who hears about it) — ascending, and inclusive.
    "escalation": ((1, "owner's director"), (31, "head of model risk"),
                   (91, "model risk committee")),
    "severity": {"CALIBRATION": "high", "STABILITY": "high", "CSI": "medium",
                 "VALIDATION-OVERDUE": "high", "NEVER-VALIDATED": "critical",
                 "AUC-FLOOR": "medium", "NOT-REPRODUCED": "critical",
                 "NOT-COMPARABLE": "medium", "NOT-REPRODUCIBLE": "high",
                 "MONITOR-BREACH": "high"},
}


def raise_findings(evidence: Mapping[str, Any], policy: Mapping[str, Any]) -> dict:
    """Turn every module's evidence into findings, keyed by source in SOURCES order. GIVEN."""
    sev = policy["severity"]
    out: dict[str, list] = {source: [] for source in SOURCES}
    cal = evidence["calibration"]
    if cal["ece"] > cal["max_ece"]:
        out["calibration"].append(Finding("CALIBRATION", "champion", sev["CALIBRATION"],
                                          f"champion ECE {cal['ece']:.4f} above the ceiling "
                                          f"of {cal['max_ece']:.4f}"))
    stab = evidence["stability"]
    if stab["psi"].psi > stab["threshold"]:
        w = stab["worst_bin"]
        out["stability"].append(Finding(
            "STABILITY", "score", sev["STABILITY"],
            f"score PSI {stab['psi'].psi:.4f} above the threshold of {stab['threshold']:.2f}; "
            f"largest contribution {stab['psi'].contributions[w]:.4f} in bin {w}"))
    for shift in stab["shifts"]:
        if shift.csi > stab["threshold"]:
            out["stability"].append(Finding(
                "CSI", shift.driver, sev["CSI"],
                f"{shift.driver} CSI {shift.csi:.4f} over {shift.bins} bins against "
                f"{stab['threshold']:.2f}; log-odds shift {shift.log_odds_shift:+.4f}"))
    tiering = evidence["tiering"]
    cov = tiering["coverage"]
    if cov.verdict == "overdue":
        out["tiering"].append(Finding(
            "VALIDATION-OVERDUE", cov.model_id, sev["VALIDATION-OVERDUE"],
            f"tier {tiering['tier'].tier} model; {cov.months_since} months since its last "
            f"validation against a frequency of {cov.due_months} months"))
    elif cov.verdict == "never_validated":
        out["tiering"].append(Finding("NEVER-VALIDATED", cov.model_id, sev["NEVER-VALIDATED"],
                                      "in use and never validated"))
    out["soundness"].extend(evidence["soundness"]["findings"])   # module 3 already rated them
    dis = evidence["discrimination"]
    for name, iv in dis["intervals"].items():
        if iv.lo < dis["min_auc"]:
            out["discrimination"].append(Finding(
                "AUC-FLOOR", name, sev["AUC-FLOOR"],
                f"AUC {iv.point:.3f} with interval {iv.lo:.3f} to {iv.hi:.3f} reaches below "
                f"the floor of {dis['min_auc']:.3f}"))
    rep = evidence["reproduction"]["verdict"]
    if rep["name"] != "REPRODUCED":     # module 6: only bit-for-bit reproduction closes it
        out["reproduction"].append(Finding(
            "NOT-REPRODUCED", "portfolio aggregate", sev["NOT-REPRODUCED"],
            f"the reproduction verdict is {rep['name']}: the second line's total differs from "
            f"the first line's by {rep['gap']:.3e} against a worst-case bound of "
            f"{rep['bound']:.3e}"))
    for name, audit in evidence["comparability"]["audits"].items():
        if not audit.comparable:
            failed = ", ".join(c.name for c in audit.checks if not c.held)
            out["comparability"].append(Finding("NOT-COMPARABLE", name, sev["NOT-COMPARABLE"],
                                                f"not comparable: {failed} not held"))
    cert = evidence["explainability"]["certificate"]
    if not cert["reproduced"]:
        first = cert["first_difference"]
        out["explainability"].append(Finding(
            "NOT-REPRODUCIBLE", "explanation pack", sev["NOT-REPRODUCIBLE"],
            f"regenerated under hash seed {first['hash_seed']}, the pack first differs from "
            f"the one in hand at line {first['line']}"))
    mon = evidence["monitoring"]["report"]
    if mon["breach"]:
        out["monitoring"].append(Finding(
            "MONITOR-BREACH", f"extract to {mon['extract_to']}", sev["MONITOR-BREACH"],
            f"month PSI {mon['psi']:.4f} above {mon['threshold']:.2f} over "
            f"{mon['records']:,} records; breach in bin {mon['breach_bin']}"))
    return {source: tuple(found) for source, found in out.items()}


RAISED = raise_findings(EVIDENCE, PACK_POLICY)
for _source, _found in RAISED.items():
    for _f in _found:
        print(f"{_source:<15} {MODULE_OF[_source]:<18} {_f.severity:<8} {_f.rule:<19} {_f.subject}")

## 4. Exercise 1 — `build_register()`

A finding raised this quarter may be the same finding the committee saw last quarter. The
register has to know, because the committee's first question about any finding is *how long
has this been open?* — and the answer lives in the date it was FIRST raised.

The interagency guidance of April 2026 notes that documentation can support the tracking of
recommendations, responses and exceptions, and help manage model remediation. The register
is where that tracking lives. A finding in it is keyed by where it came from, what rule
raised it and what it concerns. When this cycle raises a key the last pack already carried,
the entry keeps its identifier, its raised date, its due date and its owner; only its
severity and its evidence are this cycle's. A finding the last pack had CLOSED that comes
back is **reopened**, not new. And an entry the last pack carried that nobody raised this
cycle stays in the register exactly as it was: not being raised again is not the same thing
as being fixed.

The last pack's register is `HISTORY`, below. Print it before you start.

<details><summary>💡 Hint 1 — what to think about</summary>

Three things carry the marks. First, identity: what makes two findings the same finding, and
what an entry keeps when it recurs — everything that describes its history, nothing that
describes this cycle's evidence. Second, a recurrence of a closed finding is news, and its
status must say so. Third, numbering: a new finding's identifier must never collide with or
re-use an old one, so it continues from the highest identifier the last pack issued, and new
findings are numbered in the order the docstring fixes, so two runs number them identically.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Parse the as-of date first. Index the history by its three-part key. Walk this cycle's
findings source by source, rejecting a key raised twice. A key found in the history becomes
that entry with this cycle's severity and detail and the status rule applied. A new key
needs an owner for its source and a remediation period for its severity from the policy —
a ValueError naming whichever is missing — and is set aside. Carry every history entry this
cycle did not raise unchanged. Sort the new findings by the docstring's key, number them on
from the highest existing number, give each the as-of date as raised and the as-of date plus
its remediation period as due. Return everything sorted by severity, then identifier.

</details>

In [ ]:
STATUSES = ("open", "reopened", "risk accepted", "closed")
LIVE = ("open", "reopened")      # a finding the committee still has to worry about


class RegisterEntry(NamedTuple):
    finding_id: str      # "F-001" ... — stable from one pack to the next
    source: str          # the part of the suite that raised it: one of SOURCES
    rule: str            # the finding family, as the raising module names it
    subject: str         # what it concerns; "" when it concerns the model as a whole
    severity: str        # one of SEVERITY_ORDER
    detail: str          # this cycle's evidence, with its figures
    owner: str           # who must remediate it
    raised: str          # YYYY-MM-DD it was FIRST raised
    due: str             # YYYY-MM-DD remediation is due
    status: str          # one of STATUSES


# The register as the last pack, dated 2026-06-30, left it. Due dates follow PACK_POLICY.
HISTORY = (
    RegisterEntry("F-001", "stability", "STABILITY", "score", "high",
                  "score PSI above the threshold at the last pack", "retail credit modelling",
                  "2026-03-31", "2026-06-29", "open"),
    RegisterEntry("F-002", "soundness", "MONO", "months_on_book", "high",
                  "months_on_book moved the wrong way under the sweep", "retail credit modelling",
                  "2026-03-31", "2026-06-29", "closed"),
    RegisterEntry("F-003", "comparability", "NOT-COMPARABLE", "vendor-challenger", "medium",
                  "not comparable: missing values not held", "vendor model oversight",
                  "2026-03-31", "2026-09-27", "risk accepted"),
    RegisterEntry("F-004", "tiering", "VALIDATION-OVERDUE", "champion-v3", "high",
                  "validation overdue at the last pack", "model risk inventory",
                  "2026-06-30", "2026-09-28", "open"),
    RegisterEntry("F-005", "soundness", "VAR-UNUSED", "income_k", "medium",
                  "income_k is documented and is never read by the implementation",
                  "retail credit modelling", "2026-03-31", "2026-09-27", "closed"),
    RegisterEntry("F-006", "calibration", "CALIBRATION", "champion", "high",
                  "champion ECE above the ceiling at the last pack", "retail credit modelling",
                  "2026-06-30", "2026-09-28", "open"),
)


def build_register(raised: Mapping[str, tuple], history: tuple, policy: Mapping[str, Any],
                   as_of: str) -> tuple:
    """Merge this cycle's findings into the last pack's register.

    `raised` maps each source to a tuple of `Finding`s (module 3's format). `history` is the
    last pack's register, a tuple of `RegisterEntry`. A finding's KEY is
    `(source, finding.rule, finding.subject)`.

    Requirements, each graded:
      * a key already in `history` keeps that entry's `finding_id`, `raised`, `due` and
        `owner`. Its `severity` and `detail` become this cycle's. Its `status` becomes
        `"reopened"` if the history says `"closed"`, and otherwise stays what the history says
        (open, reopened and risk accepted all carry over).
      * a history entry whose key is NOT raised this cycle is carried forward unchanged.
      * a new key gets `raised = as_of`, `due = as_of + policy["days_to_remediate"][severity]`
        calendar days, `owner = policy["owners"][source]` and `status = "open"`.
      * new identifiers continue from the HIGHEST number in `history` ("F-007" after "F-006",
        whatever gaps there are), zero-padded to three digits, and are handed out in the
        order (severity rank in SEVERITY_ORDER, source position in SOURCES, rule, subject).
        With an empty history the first is "F-001".
      * the result is a tuple of `RegisterEntry`, sorted by severity rank, then finding_id.
      * `ValueError` if a severity is not in SEVERITY_ORDER, if a source is not in SOURCES,
        if a key is raised twice this cycle, or if the policy has no owner for a new
        finding's source or no remediation period for its severity — naming what is missing.
      * nothing reads the clock: every date comes from `as_of` or from `history`.

    Example:
        >>> reg = build_register({"calibration": (Finding("CALIBRATION", "champion", "high",
        ...                        "ECE above the ceiling"),)}, (), PACK_POLICY, "2026-09-30")
        >>> reg[0].finding_id, reg[0].raised, reg[0].due, reg[0].status
        ('F-001', '2026-09-30', '2026-12-29', 'open')

    Returns:
        the merged register, a tuple of RegisterEntry.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_register() -> None:
    pol = {"owners": {"calibration": "team a", "stability": "team b"},
           "days_to_remediate": {"critical": 30, "high": 90, "medium": 180, "low": 365}}
    raised = {"calibration": (Finding("CALIBRATION", "champion", "high", "ECE now"),),
              "stability": (Finding("STABILITY", "score", "critical", "PSI now"),)}
    reg = build_register(raised, (), pol, "2026-09-30")
    assert isinstance(reg, tuple) and all(isinstance(e, RegisterEntry) for e in reg), (
        "return a tuple of RegisterEntry, not a list or a dict")
    ids = {e.rule: e.finding_id for e in reg}
    assert ids == {"STABILITY": "F-001", "CALIBRATION": "F-002"}, (
        f"identifiers came out {ids} — new findings are numbered in severity order, so the "
        "critical one is F-001")
    cal = next(e for e in reg if e.rule == "CALIBRATION")
    assert (cal.raised, cal.due, cal.owner, cal.status) == ("2026-09-30", "2026-12-29",
                                                            "team a", "open"), (
        f"got raised {cal.raised}, due {cal.due}, owner {cal.owner!r}, status {cal.status!r} — "
        "a new high finding is raised on the as-of date and due 90 calendar days later")
    history = (RegisterEntry("F-004", "calibration", "CALIBRATION", "champion", "high",
                             "ECE then", "team a", "2026-03-31", "2026-06-29", "closed"),
               RegisterEntry("F-009", "tiering", "VALIDATION-OVERDUE", "m1", "high",
                             "overdue then", "team c", "2026-03-31", "2026-06-29", "open"))
    reg2 = build_register(raised, history, pol, "2026-09-30")
    cal2 = next(e for e in reg2 if e.rule == "CALIBRATION")
    assert (cal2.finding_id, cal2.raised, cal2.due) == ("F-004", "2026-03-31", "2026-06-29"), (
        f"a recurring finding came back as {cal2.finding_id}, raised {cal2.raised} — it keeps "
        "its identifier and its FIRST raised date, or its age resets every quarter")
    assert cal2.status == "reopened" and cal2.detail == "ECE now", (
        f"status {cal2.status!r}, detail {cal2.detail!r} — a closed finding that recurs is "
        "reopened, and its evidence is this cycle's")
    stab2 = next(e for e in reg2 if e.rule == "STABILITY")
    assert stab2.finding_id == "F-010", (
        f"the new finding got {stab2.finding_id} — number on from the HIGHEST identifier in "
        "the history, not from its length")
    assert any(e.finding_id == "F-009" and e.status == "open" for e in reg2), (
        "a history entry nobody raised this cycle is carried forward unchanged, not dropped")
    try:
        build_register({"monitoring": (Finding("MONITOR-BREACH", "m", "high", "x"),)}, (), pol,
                       "2026-09-30")
    except ValueError:
        pass
    else:
        raise AssertionError("a source the policy names no owner for should raise ValueError")
    print("exercise 1 looks right")

In [ ]:
_try("exercise 1", _check_register)

In [ ]:
def _show_register() -> None:
    reg = build_register(RAISED, HISTORY, PACK_POLICY, AS_OF)
    for e in reg:
        print(f"{e.finding_id}  {e.severity:<8} {e.status:<13} raised {e.raised}  due {e.due}  "
              f"{e.source}/{e.rule} {e.subject}")
    before = [h.finding_id for h in HISTORY]
    carried = sum(e.finding_id in before for e in reg)
    print(f"\n{len(reg)} entries: {carried} carried from the last pack, {len(reg) - carried} new.")


_try("the register", _show_register, needs=("exercise 1",))

## 5. Exercise 2 — `age_register()`, the remediation tracker

A register says what is wrong. A tracker says what is late. For every LIVE finding — open or
reopened — it measures the age since the finding was first raised and the days past its due
date, and it escalates by policy: the further past due, the more senior the person who hears
about it. A closed finding is not tracked, and neither is a risk-accepted one: acceptance is a
decision somebody senior already took, not a remediation that is running late.

Every day in this computation is counted from the `as_of` argument. Not from `date.today()`,
and not from the clock in any other form: a pack that reads the clock prints different ages
on different days from the same evidence, and nobody can regenerate the one the committee
actually saw.

<details><summary>💡 Hint 1 — what to think about</summary>

Decide which statuses are tracked before you compute anything. Then the boundaries: a finding
due on the as-of date is due, not late, and an escalation threshold is reached when the days
overdue equal it, not only when they exceed it. A finding raised after the as-of date is a
data-entry error — refuse it rather than reporting a negative age.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Parse the as-of date once. Check the escalation thresholds rise strictly. For each live entry,
parse its raised and due dates, refuse a raised date later than the as-of date, take the age
as the whole days between raised and as-of, and the days overdue as the days between due and
as-of when that is positive, zero otherwise. Walk the escalation levels in order and keep the
last one whose threshold the days overdue reach; with none reached, the level is the word the
docstring gives. Sort most overdue first, then by identifier.

</details>

In [ ]:
class AgedEntry(NamedTuple):
    finding_id: str
    severity: str
    owner: str
    age_days: int        # whole days since the finding was FIRST raised
    days_overdue: int    # whole days past its due date; 0 when not yet past it
    escalation: str      # the level it has reached, or "none"


def age_register(register: tuple, as_of: str, policy: Mapping[str, Any]) -> tuple:
    """Age every live finding and escalate the overdue ones.

    Requirements, each graded:
      * only entries whose status is in LIVE ("open", "reopened") appear.
      * `age_days` is `(as_of - raised).days` and `days_overdue` is `(as_of - due).days` when
        that is positive, else 0. A finding due ON the as-of date is not overdue.
      * `escalation` is the name of the LAST level in `policy["escalation"]` whose threshold
        is <= days_overdue — thresholds are inclusive — and `"none"` when no level is reached.
      * sorted by days_overdue, largest first, then by finding_id.
      * `ValueError` naming the finding if its raised date is after `as_of`, and `ValueError`
        if the escalation thresholds do not rise strictly.
      * every day is counted from `as_of`. Never from the clock.

    Example — raised 2026-03-31, due 2026-06-29, as of 2026-09-30, escalation as PACK_POLICY:
        >>> e = RegisterEntry("F-001", "stability", "STABILITY", "score", "high", "...",
        ...                   "retail credit modelling", "2026-03-31", "2026-06-29", "open")
        >>> row = age_register((e,), "2026-09-30", PACK_POLICY)[0]
        >>> row.age_days, row.days_overdue, row.escalation
        (183, 93, 'model risk committee')

    Returns:
        a tuple of AgedEntry, one per live finding.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _entry(fid: str, raised: str, due: str, status: str = "open",
           severity: str = "high") -> RegisterEntry:
    return RegisterEntry(fid, "stability", "STABILITY", fid, severity, "detail", "team",
                         raised, due, status)


def _check_ageing() -> None:
    pol = {"escalation": ((1, "director"), (31, "head of model risk"), (91, "committee"))}
    reg = (_entry("F-001", "2026-03-31", "2026-06-29"),
           _entry("F-002", "2026-09-01", "2026-09-30"),
           _entry("F-003", "2026-03-31", "2026-06-29", "closed"),
           _entry("F-004", "2026-03-31", "2026-06-29", "risk accepted"),
           _entry("F-005", "2026-07-01", "2026-08-30", "reopened"))
    rows = age_register(reg, "2026-09-30", pol)
    assert all(isinstance(r, AgedEntry) for r in rows), "return a tuple of AgedEntry"
    assert [r.finding_id for r in rows] == ["F-001", "F-005", "F-002"], (
        f"rows came out {[r.finding_id for r in rows]} — only open and reopened findings are "
        "tracked, most overdue first")
    first = rows[0]
    assert (first.age_days, first.days_overdue, first.escalation) == (183, 93, "committee"), (
        f"F-001 aged {first.age_days}, overdue {first.days_overdue}, {first.escalation!r} — "
        "count whole calendar days from the as-of date")
    assert rows[1].escalation == "head of model risk", (
        f"thirty-one days overdue reaches the threshold of 31 exactly and escalates to it; you "
        f"gave {rows[1].escalation!r} — thresholds are inclusive")
    assert (rows[2].days_overdue, rows[2].escalation) == (0, "none"), (
        "a finding due ON the as-of date is not overdue and escalates to nobody")
    far = age_register(reg[:1], "2031-01-01", pol)[0]
    want = (_dt.date(2031, 1, 1) - _dt.date(2026, 3, 31)).days
    assert far.age_days == want, (
        f"as of 2031-01-01 the age should be {want} days, got {far.age_days} — every day is "
        "counted from as_of, never from the clock")
    try:
        age_register(reg, "2026-03-30", pol)
    except ValueError:
        pass
    else:
        raise AssertionError("a finding raised after the as-of date should raise ValueError")
    try:
        fresh = age_register((_entry("F-006", "2026-09-30", "2026-12-29"),), "2026-09-30", pol)
    except ValueError:
        raise AssertionError("a finding raised ON the as-of date was refused — it is 0 days old; "
                             "only a raised date AFTER as_of is an error") from None
    assert fresh[0].age_days == 0, (
        f"a finding raised on the as-of date is 0 days old, got {fresh[0].age_days}")
    print("exercise 2 looks right")

In [ ]:
_try("exercise 2", _check_ageing)

In [ ]:
def _show_tracker() -> None:
    reg = build_register(RAISED, HISTORY, PACK_POLICY, AS_OF)
    for r in age_register(reg, AS_OF, PACK_POLICY):
        print(f"{r.finding_id}  {r.severity:<8} age {r.age_days:>3} days  "
              f"overdue {r.days_overdue:>3} days  -> {r.escalation}")


_try("the tracker", _show_tracker, needs=("exercise 1", "exercise 2"))

## 6. Exercise 3 — `executive_summary()`

The draft in section 2 opened with an opinion somebody typed. This one is assembled: the
opinion follows from the register by a rule written down before anyone looked, and every
count on the page is a count of entries in the register or rows in the tracker — returned as
DATA, alongside the sentences, so that the gate you build later can trace every one of them.

The rule: a live critical finding makes the model **NOT FIT FOR USE**; otherwise a live high
finding, or any finding escalated to the top of the escalation ladder, makes it **FIT FOR USE
WITH CONDITIONS**; otherwise it is **FIT FOR USE**. Risk-accepted and closed findings are
counted and shown, and decide nothing.

<details><summary>💡 Hint 1 — what to think about</summary>

Which entries are live decides almost every mark: a reopened finding is live, and a
risk-accepted one is not, however severe. Every severity must appear in the per-severity
counts, including the ones with none. The headline lists live findings at the two most severe
levels in the order the register already has them — do not re-sort.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Take the name of the top escalation level from the last entry of the policy's escalation
ladder. Filter the live entries once. Count them by severity in SEVERITY_ORDER, count tracker
rows with any days overdue and tracker rows at the top level, and count the risk-accepted and
closed entries. Apply the opinion rule in order. Collect the headline identifiers. Build the
four count lines exactly as the docstring spells them, then one line per headline finding,
and return everything in a Summary.

</details>

In [ ]:
OPINIONS = ("NOT FIT FOR USE", "FIT FOR USE WITH CONDITIONS", "FIT FOR USE")


class Summary(NamedTuple):
    opinion: str          # one of OPINIONS
    live: int             # open + reopened entries
    by_severity: dict     # severity -> live count, every severity in SEVERITY_ORDER, in order
    overdue: int          # tracker rows with days_overdue > 0
    escalated: int        # tracker rows at the TOP escalation level
    accepted: int         # "risk accepted" entries
    closed: int           # "closed" entries
    headline: tuple       # finding_ids of live critical and high entries, in register order
    lines: tuple          # the summary as the committee reads it


def executive_summary(register: tuple, tracker: tuple, policy: Mapping[str, Any]) -> Summary:
    """Assemble the executive summary from the register and the tracker.

    Requirements, each graded:
      * `opinion`: "NOT FIT FOR USE" if any live entry is critical; else "FIT FOR USE WITH
        CONDITIONS" if any live entry is high or any tracker row has reached the top
        escalation level (the last name in `policy["escalation"]`); else "FIT FOR USE".
      * live means status in LIVE. Risk-accepted and closed entries decide nothing.
      * `by_severity` has every severity in SEVERITY_ORDER as a key, in that order, zero
        included.
      * `lines` is exactly these four strings, then one per headline entry:
          f"Opinion: {opinion}."
          f"Live findings: {live} ({c} critical, {h} high, {m} medium, {l} low)."
          f"Overdue: {overdue}. Escalated to the {top level name}: {escalated}."
          f"Risk accepted: {accepted}. Closed: {closed}."
          f"- {finding_id} ({severity}, {source}): {detail}"
      * nothing is typed: every number in `lines` is one of the fields you return.

    Example — one live high finding, nothing overdue:
        >>> e = RegisterEntry("F-001", "calibration", "CALIBRATION", "champion", "high",
        ...                   "ECE above the ceiling", "retail credit modelling",
        ...                   "2026-09-30", "2026-12-29", "open")
        >>> s = executive_summary((e,), (), PACK_POLICY)
        >>> s.opinion, s.lines[1]
        ('FIT FOR USE WITH CONDITIONS', 'Live findings: 1 (0 critical, 1 high, 0 medium, 0 low).')

    Returns:
        a Summary.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_summary() -> None:
    pol = {"escalation": ((1, "director"), (91, "committee"))}
    reg = (_entry("F-001", "2026-03-31", "2026-06-29", "risk accepted", "critical"),
           _entry("F-002", "2026-03-31", "2026-06-29", "reopened", "high"),
           _entry("F-003", "2026-09-01", "2026-12-30", "open", "medium"),
           _entry("F-004", "2026-03-31", "2026-06-29", "closed", "high"))
    tracker = (AgedEntry("F-002", "high", "team", 183, 93, "committee"),
               AgedEntry("F-003", "medium", "team", 29, 0, "none"))
    s = executive_summary(reg, tracker, pol)
    assert isinstance(s, Summary), "return a Summary"
    assert s.opinion == "FIT FOR USE WITH CONDITIONS", (
        f"opinion {s.opinion!r} — the critical finding is risk accepted, so it decides nothing; "
        "the live high finding makes it 'with conditions'")
    assert (s.live, s.by_severity) == (2, {"critical": 0, "high": 1, "medium": 1, "low": 0}), (
        f"live {s.live}, by severity {s.by_severity} — count open and reopened entries only, "
        "and keep every severity as a key")
    assert (s.overdue, s.escalated, s.accepted, s.closed) == (1, 1, 1, 1), (
        f"overdue {s.overdue}, escalated {s.escalated}, accepted {s.accepted}, closed "
        f"{s.closed} — overdue and escalated come from the tracker, the others from statuses")
    assert s.lines[:4] == ("Opinion: FIT FOR USE WITH CONDITIONS.",
                           "Live findings: 2 (0 critical, 1 high, 1 medium, 0 low).",
                           "Overdue: 1. Escalated to the committee: 1.",
                           "Risk accepted: 1. Closed: 1."), (
        f"the first four lines were {s.lines[:4]} — match the docstring's formats exactly")
    assert s.headline == ("F-002",) and s.lines[4] == "- F-002 (high, stability): detail", (
        f"headline {s.headline} — live critical and high entries only, each with its detail")
    worse = executive_summary(reg[1:] + (_entry("F-009", "2026-09-30", "2026-10-30", "open",
                                                "critical"),), (), pol)
    assert worse.opinion == "NOT FIT FOR USE", "a LIVE critical finding makes it NOT FIT FOR USE"
    print("exercise 3 looks right")

In [ ]:
_try("exercise 3", _check_summary)

In [ ]:
def _show_summary() -> None:
    reg = build_register(RAISED, HISTORY, PACK_POLICY, AS_OF)
    summary = executive_summary(reg, age_register(reg, AS_OF, PACK_POLICY), PACK_POLICY)
    typed = next(line for line in DRAFT.splitlines() if line.startswith("Opinion:"))
    print("the draft, typed above the findings:   " + typed)
    print("assembled from the findings:           " + summary.lines[0])
    print()
    print("\n".join(summary.lines))


_try("the summary", _show_summary, needs=("exercise 1", "exercise 2", "exercise 3"))

## 7. Exercise 4 — `limitations()`

The interagency guidance the US banking agencies issued on 17 April 2026 says that sound
validation identifies model limitations and errors. The honest place to start is the list of
things the suite could NOT measure — each one a question the committee might assume was
answered:

- a subgroup below minimum support, whose metrics module 1 suppressed;
- reliability bins with no observations in them, where calibration was never tested;
- a comparison module 4 called NOT MEASURABLE: the sample could not have detected a material
  difference, so the interval is too wide to conclude anything. NOT SIGNIFICANT is a
  different verdict — that sample could have seen a difference and did not — and it is a
  result, not a limitation.

<details><summary>💡 Hint 1 — what to think about</summary>

Two traps. Adjacent empty bins are one gap in the evidence, not several, so a run of them
becomes a single limitation spanning from the lower edge of its first bin to the upper edge of
its last — while two runs separated by an occupied bin stay two. And only one of module 4's
three verdicts is a limitation.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Walk the subgroup table and emit one limitation for each group that is not reportable, with
its true count and the minimum support. For each named reliability table, walk its counts,
find each maximal run of empty bins and emit one limitation per run. For each named comparison,
emit one only when its verdict is the not-measurable one, using the material difference from
the discrimination evidence. Keep the kinds in the docstring's order and each kind's items in
the order the evidence gives them.

</details>

In [ ]:
LIMITATION_KINDS = ("suppressed subgroup", "empty bins", "not measurable")


class Limitation(NamedTuple):
    kind: str        # one of LIMITATION_KINDS
    subject: str     # the group, the table or the comparison it concerns
    detail: str


def limitations(evidence: Mapping[str, Any]) -> tuple:
    """List what the suite could not measure, from the evidence.

    Reads `evidence["calibration"]["subgroups"]` (module 1's table), `["min_support"]` and
    `["reliability"]` (a dict of name -> ReliabilityTable), and
    `evidence["discrimination"]["comparisons"]` (a dict of name -> Measurability) and
    `["material"]`.

    Requirements, each graded, with each `detail` EXACTLY as shown:
      * "suppressed subgroup", one per group that is not reportable, subject = group name:
          f"{count} records against a minimum support of {min_support}; no metric is
          reported for it"
      * "empty bins", one per maximal run of consecutive bins with count 0, subject = table
        name: f"no observations between {lo:.2f} and {hi:.2f}", where lo is the run's first
        bin's lower edge and hi its last bin's upper edge.
      * "not measurable", one per comparison whose verdict is "NOT MEASURABLE", subject =
        comparison name: f"minimum detectable difference {mdd:.4f} against a material
        difference of {material:.4f}; {n_required:,} records would be needed"
      * the kinds in LIMITATION_KINDS order; within a kind, the order of the evidence.
      * evidence with nothing unmeasured returns `()`.

    Example — bins [5, 0, 0, 3, 0] of width 0.2 give two runs:
        "no observations between 0.20 and 0.60" and "no observations between 0.80 and 1.00"

    Returns:
        a tuple of Limitation.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_limitations() -> None:
    edges = np.linspace(0.0, 1.0, 6)
    table = ReliabilityTable(edges[:-1], edges[1:], np.array([5, 0, 0, 3, 0]),
                             np.full(5, np.nan), np.full(5, np.nan))
    sub = {"group": np.array(["big", "tiny"]), "count": np.array([900, 37]),
           "reportable": np.array([True, False])}
    comps = {"wide": Measurability(0.0312, 14321, "NOT MEASURABLE", "..."),
             "seen": Measurability(0.0050, 900, "NOT SIGNIFICANT", "..."),
             "found": Measurability(0.0050, 900, "SIGNIFICANT", "...")}
    ev = {"calibration": {"subgroups": sub, "min_support": 200, "reliability": {"t": table}},
          "discrimination": {"comparisons": comps, "material": 0.01}}
    got = limitations(ev)
    assert isinstance(got, tuple) and all(isinstance(x, Limitation) for x in got), (
        "return a tuple of Limitation")
    kinds = [x.kind for x in got]
    assert kinds == ["suppressed subgroup", "empty bins", "empty bins", "not measurable"], (
        f"kinds came out {kinds} — one suppressed group, two separate runs of empty bins, and "
        "ONE not-measurable comparison: NOT SIGNIFICANT is a result, not a limitation")
    assert got[0].detail == ("37 records against a minimum support of 200; no metric is "
                             "reported for it"), f"suppressed-group detail was {got[0].detail!r}"
    assert got[1].detail == "no observations between 0.20 and 0.60", (
        f"got {got[1].detail!r} — two adjacent empty bins are ONE run, from the first bin's "
        "lower edge to the last bin's upper edge")
    assert got[3].detail == ("minimum detectable difference 0.0312 against a material "
                             "difference of 0.0100; 14,321 records would be needed"), (
        f"not-measurable detail was {got[3].detail!r}")
    print("exercise 4 looks right")

In [ ]:
_try("exercise 4", _check_limitations)

In [ ]:
def _show_limitations() -> None:
    for x in limitations(EVIDENCE):
        print(f"- {x.kind} — {x.subject}: {x.detail}")


_try("the limitations", _show_limitations, needs=("exercise 4",))

## 8. What counts as a figure — exercise 5, `extract_figures()`

Everything above produces a document. The rest of the lesson makes the document prove
itself, and it starts with a definition, because "every number must trace" is not a rule
until you can say which strings are numbers. Get it wrong one way and the gate drowns the
committee secretary in false alarms — a date is not a result, and neither is a section number,
a finding identifier or the `3` in `champion-v3`. Get it wrong the other way and the typed
number walks through.

The definition this course uses, which your function implements exactly:

- **Not figures:** a date written `YYYY-MM-DD`; the section number that opens a markdown
  heading; any number inside a *word* — a run of letters, digits, `_`, `+`, `-` and `.` —
  that also contains a letter (`F-003`, `champion-v3`, `S1`, `A.1`). Such a word is a name.
- **Except** a word that is a number in scientific notation (`6.836e-03`): that is a figure.
- **How a figure is read:** `1,234` is one figure, not two; a `-` or `+` straight before the
  digits is a sign unless a digit stands right before it, so `0.00-0.10` is two positive
  figures; `12.5%` means 0.125.
- **How precise it claims to be:** a figure printed to *d* decimal places claims its value to
  within half a unit in that last place. That half unit travels with it; it is the only
  honest tolerance, because it is the one the document itself asserted.

<details><summary>💡 Hint 1 — what to think about</summary>

Do it in two passes per line: first decide which character spans are names rather than
figures — dates, the heading's section number, identifier words — then find every number and
keep only those that do not start inside such a span. The sign and the percent sign are
decisions about the characters immediately either side of the digits, and the precision comes
from counting digits after the decimal point, adjusted for an exponent and for a percent.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Number the lines from one. For each line, collect masked spans: every date match; on a line
that starts with hashes, the section token after them; every word containing both a letter and
a digit unless, with any trailing full stop removed, it is wholly a number in scientific
notation. Then scan with one pattern that allows an optional sign (only where the preceding
character is not a digit), digits with optional comma groups of exactly three, an optional
fraction, an optional exponent and an optional percent. Skip matches whose digits start in a
masked span. Read the value with the commas removed, divide by a hundred for a percent, and
build the half unit from the count of fraction digits and the exponent.

</details>

In [ ]:
class Figure(NamedTuple):
    text: str          # exactly as printed: "1,234", "12.5%", "-0.0123", "6.836e-03"
    value: float       # what it means: 1234.0, 0.125, -0.0123, 0.006836
    half_unit: float   # half a unit in its last printed place, on the same scale as value
    line: int          # 1-based line number in the document


def extract_figures(text: str) -> tuple:
    """Every figure in `text`, under this course's definition, in document order.

    Requirements, each graded:
      * NOT figures: a date `YYYY-MM-DD`; the section number straight after the `#`s of a
        markdown heading line ("## 3." , "### 3.2", "## A.1"); any number inside a word —
        a maximal run of `[A-Za-z0-9_.+-]` — that contains at least one letter, UNLESS that
        word, minus trailing full stops, is wholly a number in scientific notation
        ("6.836e-03", "2.000e+05").
      * a figure is: an optional sign, then digits (with `,` only between groups of exactly
        three: "4,194,304"), an optional fraction, an optional exponent (`e-03`), and an
        optional `%` directly after. ".5" is a figure too.
      * the sign counts only when the character before it is not a digit: in "0.00-0.10"
        both figures are positive, and "=-0.0588" is negative.
      * `value`: commas removed, sign applied, and divided by 100 for a `%`.
      * `half_unit`: 0.5 * 10 ** (exponent - decimals), where decimals counts the digits
        after the point (0 for an integer) and exponent is 0 without one; divided by 100
        for a `%`. So "0.0727" -> 5e-05, "6,000" -> 0.5, "22.6%" -> 0.0005.
      * `line` is 1-based; figures come back in document order as a tuple of Figure.

    Example:
        >>> [(f.text, f.value) for f in extract_figures("F-003 on 2026-09-30: 1,234 and 12.5%")]
        [('1,234', 1234.0), ('12.5%', 0.125)]

    Returns:
        a tuple of Figure.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_figures() -> None:
    got = extract_figures("Opinion: 6,000 records at 22.6%, ECE 0.0727, change -0.0588.")
    assert all(isinstance(f, Figure) for f in got), "return a tuple of Figure"
    assert [f.text for f in got] == ["6,000", "22.6%", "0.0727", "-0.0588"], (
        f"figures came out {[f.text for f in got]} — '6,000' is ONE figure, and the sign and "
        "the percent sign belong to the figure they touch")
    assert np.allclose([f.value for f in got], [6000.0, 0.226, 0.0727, -0.0588],
                       rtol=0.0, atol=1e-12), (
        f"values {[f.value for f in got]} — commas removed, 22.6% means 0.226")
    assert abs(got[1].half_unit - 0.0005) < 1e-15 and got[0].half_unit == 0.5, (
        f"half units {[f.half_unit for f in got]} — '22.6%' claims 0.226 to within 0.0005, and "
        "'6,000' to within 0.5")
    names = extract_figures("# Committee pack — champion-v3\n\n## 3. Findings register\n"
                            "F-003 raised 2026-03-31, rated S1.")
    assert names == (), (
        f"found {[f.text for f in names]} in a document holding no results — dates, section "
        "numbers, identifiers and rating codes are names, not figures")
    ranged = extract_figures("| 0.00-0.10 | 0.0471 |")
    assert [f.value for f in ranged] == [0.0, 0.1, 0.0471], (
        f"got {[f.value for f in ranged]} — in '0.00-0.10' the dash follows a digit, so it is "
        "a range, not a minus sign")
    sci = extract_figures("differs by 6.836e-03.")
    assert len(sci) == 1 and abs(sci[0].value - 0.006836) < 1e-15, (
        "a word in scientific notation is a figure, not an identifier")
    assert abs(sci[0].half_unit - 5e-7) < 1e-20, (
        f"'6.836e-03' claims its value to within half of 0.001e-03, got {sci[0].half_unit}")
    assert extract_figures("## 4. Limitations (5 found)")[0].value == 5.0, (
        "only the section number of a heading is a name; other numbers on it are figures")
    print("exercise 5 looks right")

In [ ]:
_try("exercise 5", _check_figures)

In [ ]:
def _show_naive_extraction() -> None:
    naive = [(n, tok) for n, line in enumerate(DRAFT.splitlines(), start=1)
             for tok in re.findall(r"-?\d+(?:\.\d+)?", line)]
    yours = extract_figures(DRAFT)
    print(f"the draft pack: {len(naive)} numbers by the naive pattern -?\\d+(\\.\\d+)?, "
          f"{len(yours)} figures by your definition")
    kept = {(f.line, f.text.replace(",", "").rstrip("%")) for f in yours}
    extra = [(n, tok) for n, tok in naive if (n, tok) not in kept]
    lines = DRAFT.splitlines()
    print(f"{len(extra)} naive readings your definition rejects or reads differently; the first "
          "twelve:")
    for n, tok in extra[:12]:
        print(f"  line {n:>3}  {tok!r:>9}  in  {lines[n - 1][:64]!r}")


_try("naive extraction", _show_naive_extraction, needs=("exercise 5",))

## 9. Exercise 6 — `computed_values()`

A figure traces to a *computed result*. So the gate needs every result the run produced, in
one list, each with the path that locates it — `validation.psi.psi`,
`tracker[0].days_overdue` — so a traced figure can be shown to an auditor with its source
beside it.

Two things in the findings are not results, and treating them as results is how a gate
quietly stops working. A **string** is not a result, even when it holds digits: the commentary
a person typed is a string in the findings, and if its digits counted, a typed number would
trace to itself. A **boolean** is not a result either: Python calls `True` an integer equal to
1, and a flag is not a count.

<details><summary>💡 Hint 1 — what to think about</summary>

This is a recursive walk over whatever the pipeline produced, and each kind of container has
its own idea of a name for its children: mappings have keys, named tuples have field names,
plain sequences and arrays have positions. Decide the order of the tests carefully — a named
tuple IS a tuple, and a boolean IS an integer, so the more specific test has to come first.
NaN and infinity can never be printed as a figure, so they are not candidates either.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Write an inner function taking an object and the path so far. Return at once for booleans of
either kind, strings, bytes and None. For a Python or numpy integer or float, convert to a
float and record it with its path when it is finite. For a numpy array, skip a boolean array
and otherwise walk every element with its index in square brackets. For a mapping, walk each
value in insertion order with a dot and the key; for a named tuple, each field with a dot and
its name; for any other list or tuple, each item with its position in square brackets. A
child of the root has no leading dot.

</details>

In [ ]:
def computed_values(findings: Any) -> tuple:
    """Every computed result inside `findings`, as `(path, value)` pairs, in walk order.

    Requirements, each graded:
      * a result is a Python or numpy integer or float, converted to `float`, and finite.
      * NOT results: `bool` and `numpy.bool_` (a flag is not a count), strings and bytes
        (even full of digits), None, NaN and infinities, and anything of another type.
      * the walk is DEPTH FIRST — everything inside one child comes before the next child —
        and descends into mappings (values in insertion order; keys are labels, not
        results), named tuples (fields in order), lists and tuples, and numpy arrays (every
        element in index order; a boolean array is skipped whole). The gate takes the FIRST
        value that matches, so the order is part of the answer.
      * paths: a mapping key or a field name is joined with "." ("validation.psi.psi"), a
        position with square brackets ("tracker[0].days_overdue"); an element of a
        multi-dimensional array is "[i,j]". A child of the root has no leading ".".

    Example:
        >>> computed_values({"a": 1, "b": {"c": [0.5, True, "7"]}})
        (('a', 1.0), ('b.c[0]', 0.5))

    Returns:
        a tuple of (path, float) pairs.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_values() -> None:
    got = computed_values({"a": 1, "b": {"c": [0.5, True, "7"]}})
    assert isinstance(got, tuple) and got == (("a", 1.0), ("b.c[0]", 0.5)), (
        f"got {got} — the docstring example: True is a flag and '7' is a string, and neither "
        "is a computed result")
    st = StabilityResult(0.25, np.array([0.1, 0.15]), np.array([0.5, 0.5]),
                         np.array([0.4, 0.6]))
    paths = dict(computed_values({"psi": st, "n": np.int64(3), "gap": np.float64(-0.5),
                                  "flag": np.bool_(True), "mask": np.array([True, False]),
                                  "missing": float("nan")}))
    assert paths.get("psi.contributions[1]") == 0.15 and paths.get("psi.psi") == 0.25, (
        f"paths found: {sorted(paths)} — a named tuple's fields are named with '.', array "
        "elements with their index in brackets")
    assert paths.get("n") == 3.0 and paths.get("gap") == -0.5, (
        "numpy integers and floats are results too")
    assert "flag" not in paths and not any(p.startswith("mask") for p in paths), (
        "numpy booleans and boolean arrays are flags, not results")
    assert "missing" not in paths, "NaN is never a figure anybody printed; leave it out"
    print("exercise 6 looks right")

In [ ]:
_try("exercise 6", _check_values)

## 10. Exercise 7 — `trace_gate()`

Now the gate. For every figure in the document, look for a computed result within that
figure's own half unit — the precision the document claimed — and record the first path that
matches. A figure with no such result is **untraceable**, and one untraceable figure fails the
document.

Resist every temptation to make it pass. A gate that tolerates a bit more than half a unit
passes a figure copied one digit wrong. A gate that skips integers, or percentages, or tables,
or headings, passes a typed number of that kind. And a gate whose values include strings
passes a typed number by finding it in the very text that typed it. Each of those gates is
quieter; none of them is a gate.

The one allowance is for binary arithmetic: a result sitting exactly half a unit from its
printed figure is legitimately rounded either way, so the comparison allows `TRACE_SLACK`, a
relative one part in a billion, on the half unit.

<details><summary>💡 Hint 1 — what to think about</summary>

Everything the gate needs is already built: the figures and their half units, and the values
with their paths. The work is the comparison, and the discipline is that the tolerance comes
from the figure — never from a constant you chose — and that nothing about a figure's kind,
position or size exempts it. Look the two helpers up when the gate is called, so a caller can
hand in different ones.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Use the extractor and the collector you were given, falling back to your own functions when
none were passed. Extract the figures, collect the values once, and for each figure take the
first value whose distance from the figure is no more than the figure's half unit enlarged by
the slack. Record the figure with that value's path, or record it as untraced. The document
passes only when nothing is untraced.

</details>

In [ ]:
TRACE_SLACK = 1e-9


class GateResult(NamedTuple):
    passed: bool          # True only when every figure traced
    n_figures: int        # figures checked
    traced: tuple         # ((Figure, path), ...) in document order
    untraced: tuple       # (Figure, ...) in document order


def trace_gate(report: str, findings: Any, extract: Callable | None = None,
               collect: Callable | None = None) -> GateResult:
    """Trace every figure in `report` to a computed result in `findings`, or fail it.

    `extract` defaults to `extract_figures` and `collect` to `computed_values`, both looked up
    when the gate is CALLED, so a caller can pass in its own.

    Requirements, each graded:
      * a figure traces to a value when `abs(value - figure.value) <= figure.half_unit *
        (1 + TRACE_SLACK)`. Nothing wider: not a fixed tolerance, not a relative one.
      * it traces to the FIRST such value in `collect(findings)` order, and `traced` records
        `(figure, path)`.
      * every figure is checked — integers, percentages, negatives, figures in tables and in
        headings alike.
      * `passed` is True exactly when `untraced` is empty; `n_figures` counts every figure.

    Example:
        >>> r = trace_gate("ECE 0.0727, n = 6,000", {"ece": 0.072691, "n": 6000})
        >>> r.passed, [path for _, path in r.traced]
        (True, ['ece', 'n'])

    Returns:
        a GateResult.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_gate() -> None:
    ok = trace_gate("ECE 0.0727, n = 6,000", {"ece": 0.072691, "n": 6000})
    assert isinstance(ok, GateResult), "return a GateResult"
    assert ok.passed and [p for _, p in ok.traced] == ["ece", "n"], (
        f"the docstring example should pass, tracing to ['ece', 'n']; got passed={ok.passed}, "
        f"untraced {[f.text for f in ok.untraced]}")
    assert trace_gate("rate 22.6%", {"rate": 0.22598}).passed, (
        "'22.6%' means 0.226 and traces to 0.22598 — compare on the figure's own scale")
    slip = trace_gate("ECE 0.0728", {"ece": 0.072691})
    assert not slip.passed and [f.text for f in slip.untraced] == ["0.0728"], (
        "0.072691 prints as 0.0727; a document saying 0.0728 is one digit wrong and must fail — "
        "is your tolerance wider than the figure's half unit?")
    typed = trace_gate("7 findings are open", {"open": 6, "note": "7 findings are open"})
    assert not typed.passed, (
        "a count of 7 that no computed result holds must fail, even though a string in the "
        "findings says it — strings are not results, and integers are not exempt")
    assert not trace_gate("gap -0.0123", {"gap": 0.0123}).passed, (
        "-0.0123 does not trace to +0.0123: a sign is a direction")
    assert trace_gate("", {}).n_figures == 0, "an empty document has no figures to check"
    print("exercise 7 looks right")

In [ ]:
_try("exercise 7", _check_gate)

In [ ]:
def _show_gate_on_draft() -> None:
    result = trace_gate(DRAFT, VALIDATION)
    print(f"the draft pack: {result.n_figures} figures, {len(result.traced)} traced, "
          f"{len(result.untraced)} untraceable")
    values = computed_values(VALIDATION)
    for fig in result.untraced:
        path, value = min(values, key=lambda pv: abs(pv[1] - fig.value))
        print(f"  UNTRACEABLE {fig.text!r} on line {fig.line}: the nearest computed result is "
              f"{path} = {value:.6f},\n  {abs(value - fig.value) / fig.half_unit:.1f} half-units "
              "away. Nothing the run produced prints as this figure.")
    for fig, path in result.traced[:4]:
        print(f"  traced {fig.text!r:>10} on line {fig.line:>2} -> {path}")
    print("  (each figure traces to the FIRST result that prints as it, which is not always the "
          "one\n  its sentence means — section 13 comes back to that)")


def _show_tolerances() -> None:
    figures = extract_figures(DRAFT)
    values = np.array([v for _, v in computed_values(VALIDATION)])

    def flagged(value: float, tol: float) -> bool:
        return bool(np.min(np.abs(values - value)) > tol)

    rules = (("half a unit (the definition)", lambda f: f.half_unit * (1 + TRACE_SLACK)),
             ("fixed 0.0001", lambda f: 1e-4), ("fixed 0.001", lambda f: 1e-3),
             ("fixed 0.01", lambda f: 1e-2), ("one per cent of the figure",
                                              lambda f: 0.01 * abs(f.value)))
    print("each gate on the draft, and on every traced figure copied one digit high in its "
          "last place:")
    for name, tol in rules:
        on_draft = sum(flagged(f.value, tol(f)) for f in figures)
        slips = [f for f in figures if not flagged(f.value, f.half_unit * (1 + TRACE_SLACK))]
        caught = sum(flagged(f.value + 2 * f.half_unit, tol(f)) for f in slips)
        print(f"  {name:<30} flags {on_draft} figure(s) on the draft · catches {caught} of "
              f"{len(slips)} one-digit slips")


_GATE_EXERCISES = ("exercise 5", "exercise 6", "exercise 7")
_try("the gate on the draft", _show_gate_on_draft, needs=_GATE_EXERCISES)
_try("tolerances", _show_tolerances, needs=_GATE_EXERCISES)

## 11. Exercise 8 — `render_committee_pack()`

The artefact. It extends module 1's generator: the executive summary, the findings register,
the remediation tracker, the limitations, the commentary a person may add, and then module 1's
report itself — and before it hands the text to anyone, it runs the gate over the WHOLE of it,
commentary included, and refuses to render while any figure is untraceable.

The commentary is the one place a person still writes words into the pack, and so it is the
one place a typed number can still get in. It is printed verbatim. Dropping it, or gating the
pack before it is added, would make the refusal disappear — and the typed number with it.

<details><summary>💡 Hint 1 — what to think about</summary>

This function reports; it measures nothing. Every figure it prints is formatted out of
findings, and the order of operations is the graded part: build the complete text, then gate
that exact text, then either return it or raise. A refusal is an exception naming the
untraceable figures, never a warning line inside a pack that renders anyway.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Check for every required key and name all the missing ones in one ValueError. Build a list of
lines: the title and as-of line, then each heading of PACK_SECTIONS in turn followed by its
content — the summary's lines, the register table and a detail bullet per entry, the tracker
table or its empty sentence, a bullet per limitation or its empty sentence, the commentary or
its empty sentence, and module 1's report without its title line and with each heading pushed
one level down. Join with newlines, gate the joined text with the gate you were given or your
own, and raise the given exception with the untraced figures when it fails.

</details>

In [ ]:
PACK_SECTIONS = ("## 1. Executive summary", "## 2. Findings register",
                 "## 3. Remediation tracker", "## 4. Limitations", "## 5. Commentary",
                 "## 6. Validation report")
PACK_KEYS = ("model_id", "as_of", "validation", "register", "tracker", "summary",
             "limitations", "commentary")


class UntraceableFigures(ValueError):
    """Raised by the pack when a figure in it traces to no computed result. GIVEN."""

    def __init__(self, figures) -> None:
        self.figures = tuple(figures)
        listed = "; ".join(f"{f.text!r} on line {f.line}" for f in self.figures)
        super().__init__(f"the pack will not render: {len(self.figures)} figure(s) trace to "
                         f"no computed result: {listed}")


def render_committee_pack(findings: Mapping[str, Any], gate: Callable | None = None) -> str:
    """Render the committee pack from `findings`, and refuse while any figure is untraceable.

    `findings` carries every key in PACK_KEYS: `validation` is module 1's findings dict,
    `register` a tuple of RegisterEntry, `tracker` a tuple of AgedEntry, `summary` a Summary,
    `limitations` a tuple of Limitation, `commentary` a string. `gate` defaults to
    `trace_gate`, looked up when called.

    The text, each line graded:
      * line 1 `f"# Committee pack — {model_id}"`; then a blank line; then
        `f"As of {as_of}. Every figure in this pack was traced to a computed result before it
        rendered."`; then the six PACK_SECTIONS headings in order, each followed by a blank
        line, its content, and a blank line.
      * 1: the summary's `lines`, verbatim, one per line.
      * 2: the header `| id | severity | source | rule | subject | owner | raised | due |
        status |`, the separator `| --- |` nine times, then one row per entry in register
        order: `| F-007 | S1 critical | reproduction | NOT-REPRODUCED | portfolio aggregate |
        credit risk reporting | 2026-09-30 | 2026-10-30 | open |` — the severity preceded by
        its SEVERITY_CODES code, and "—" for an empty subject. Then a blank line and one
        `f"- {finding_id}: {detail}"` per entry, in register order.
      * 3: `| id | severity | owner | age (days) | overdue (days) | escalation |`, the
        separator, and one row per tracker entry in tracker order; or `No live findings.`
      * 4: `f"- {kind} — {subject}: {detail}"` per limitation; or `None: the suite measured
        everything it was asked to.`
      * 5: the commentary verbatim; or `No commentary.` when it is empty.
      * 6: `render_validation_report(findings["validation"])` without its first line, every
        line that starts with `#` given one more `#`.
      * the complete text — commentary included — is passed to `gate(text, findings)`. When
        the result has not passed, raise `UntraceableFigures(result.untraced)`; otherwise
        return the text.
      * `ValueError` naming every missing key if `findings` is incomplete.

    Example — the lab's findings, once the commentary quotes its figure from them:
        >>> render_committee_pack(findings).splitlines()[0]          # doctest: +SKIP
        '# Committee pack — champion-v3 (retail application scorecard)'
        >>> render_committee_pack(findings_with_a_typed_figure)       # doctest: +SKIP
        Traceback (most recent call last):
        UntraceableFigures: the pack will not render: 1 figure(s) trace to no computed result: ...

    Returns:
        the pack, one markdown string — only when every figure in it traced.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _probe_validation() -> dict:
    """A tiny module-1 findings dict with unmistakable figures."""
    rel = ReliabilityTable(np.array([0.0, 0.5]), np.array([0.5, 1.0]), np.array([4, 0]),
                           np.array([0.25, np.nan]), np.array([0.5, np.nan]))
    psi = StabilityResult(0.7777, np.array([0.1, 0.6777]), np.array([0.5, 0.5]),
                          np.array([0.2, 0.8]))
    sub = {"group": np.array(["big"]), "count": np.array([900]), "event_rate": np.array([0.3]),
           "mean_pred": np.array([0.35]), "gap": np.array([0.05]), "ece": np.array([0.0505]),
           "reportable": np.array([True])}
    return {"model_id": "probe-model", "as_of": "2026-09-30", "data_note": "PROBE NOTE",
            "reliability": rel, "ece": 0.4242, "psi": psi, "psi_threshold": 0.25,
            "subgroups": sub, "min_support": 200, "champion": {"auc": 0.616, "ece": 0.5252},
            "challenger": {"auc": 0.818, "ece": 0.0303},
            "decision": Decision("hold", ("PASS probe rule",), {"auc_gain": 0.202}),
            "psi_worst_bin": 1}


def _probe_pack() -> dict:
    reg = (RegisterEntry("F-007", "reproduction", "NOT-REPRODUCED", "portfolio aggregate",
                         "critical", "differs by 6.836e-03", "credit risk reporting",
                         "2026-09-30", "2026-10-30", "open"),)
    tracker = (AgedEntry("F-007", "critical", "credit risk reporting", 0, 0, "none"),)
    summary = Summary("NOT FIT FOR USE", 1, {"critical": 1, "high": 0, "medium": 0, "low": 0},
                      0, 0, 0, 0, ("F-007",), ("Opinion: NOT FIT FOR USE.", "Live findings: 1."))
    return {"model_id": "probe-model", "as_of": "2026-09-30", "validation": _probe_validation(),
            "register": reg, "tracker": tracker, "summary": summary,
            "limitations": (Limitation("empty bins", "t", "no observations between 0.50 and "
                                       "1.00"),),
            "commentary": "", "gap": 0.0068359375}


def _gate_passes(text: str, findings: Any) -> GateResult:
    return GateResult(True, 0, (), ())


def _gate_refuses(text: str, findings: Any) -> GateResult:
    return GateResult(False, 1, (), (Figure("0.0140", 0.014, 5e-05, 3),))


def _check_pack() -> None:
    seen: list = []

    def spy(text: str, findings: Any) -> GateResult:
        seen.append(text)
        return GateResult(True, 0, (), ())

    probe = dict(_probe_pack(), commentary="Reviewed by the second line.")
    text = render_committee_pack(probe, gate=spy)
    lines = text.splitlines()
    assert lines[0] == "# Committee pack — probe-model", f"first line was {lines[0]!r}"
    positions = [text.find(h) for h in PACK_SECTIONS]
    assert all(p >= 0 for p in positions) and positions == sorted(positions), (
        "the six PACK_SECTIONS headings must all appear, in order")
    assert ("| F-007 | S1 critical | reproduction | NOT-REPRODUCED | portfolio aggregate | "
            "credit risk reporting | 2026-09-30 | 2026-10-30 | open |") in lines, (
        "the register row must match the docstring's format, severity code included")
    assert "- F-007: differs by 6.836e-03" in lines, "each entry's detail follows the table"
    assert "Reviewed by the second line." in lines, "the commentary is printed verbatim"
    assert "### 2. Calibration" in lines and "# Validation report — probe-model" not in lines, (
        "module 1's report follows, without its title and with every heading one level down")
    assert seen == [text], (
        "the gate must be called exactly once, on the COMPLETE text you return — commentary "
        "included")
    try:
        render_committee_pack(probe, gate=_gate_refuses)
    except UntraceableFigures as exc:
        assert "0.0140" in str(exc), "the refusal names the untraceable figure"
    else:
        raise AssertionError("a gate that fails must stop the pack: raise UntraceableFigures, "
                             "do not return the text")
    try:
        render_committee_pack({k: v for k, v in probe.items() if k != "tracker"},
                              gate=_gate_passes)
    except ValueError as exc:
        assert "tracker" in str(exc), f"the error should name the missing key, got {exc}"
    else:
        raise AssertionError("incomplete findings should raise ValueError")
    print("exercise 8 looks right")

In [ ]:
_try("exercise 8", _check_pack)

## 12. Render the pack

Everything is built. The cell below assembles the findings — module 1's run, every module's
evidence, your register, tracker, summary and limitations — and renders the pack twice. The
first time, the model owner's management response is included exactly as it was typed into
the draft. The second time, the response quotes its figure the only way a figure may enter
this pack: formatted out of the findings.

In [ ]:
DRAFT_COMMENTARY = ("Management response (model owner): the challenger improves calibration to "
                    "0.0140 and is recommended for promotion.")


def assemble_findings(commentary: str) -> dict:
    """Everything the pack reports, computed in one place, handed to one renderer. GIVEN."""
    register = build_register(RAISED, HISTORY, PACK_POLICY, AS_OF)
    tracker = age_register(register, AS_OF, PACK_POLICY)
    return {"model_id": MODEL_ID, "as_of": AS_OF, "seed": SEED, "validation": VALIDATION,
            "evidence": EVIDENCE, "policy": PACK_POLICY, "register": register,
            "tracker": tracker, "summary": executive_summary(register, tracker, PACK_POLICY),
            "limitations": limitations(EVIDENCE), "commentary": commentary}


def _show_refusal() -> None:
    try:
        render_committee_pack(assemble_findings(DRAFT_COMMENTARY))
    except UntraceableFigures as exc:
        print("REFUSED —", exc)
    else:
        raise AssertionError("the pack rendered with a typed figure in its commentary — your "
                             "gate, or your renderer's use of it, let the typed number through")


def _show_string_harvest() -> None:
    def harvest(findings: Any) -> tuple:     # a collector that ALSO reads digits out of strings
        found = list(computed_values(findings))
        found += [("commentary (a string)", f.value)
                  for f in extract_figures(findings["commentary"])]
        return tuple(found)

    findings = assemble_findings(DRAFT_COMMENTARY)
    fooled = trace_gate(render_committee_pack(findings, gate=_gate_passes), findings,
                        collect=harvest)
    print(f"a gate that reads numbers out of strings: passed={fooled.passed}, and it traced "
          f"the typed figure to {next(p for f, p in fooled.traced if f.text == '0.0140')!r}")


def _show_pack() -> None:
    chal = VALIDATION["challenger"]["ece"]
    clean = ("Management response (model owner): the challenger improves calibration to "
             f"{chal:.4f} and is recommended for promotion.")
    findings = assemble_findings(clean)
    pack = render_committee_pack(findings)
    result = trace_gate(pack, findings)
    print(pack)
    print(f"\n{result.n_figures} figures in the pack, every one traced to a computed result.")


_try("refusal", _show_refusal, needs=tuple(_EXERCISES))
_try("string harvest", _show_string_harvest, needs=tuple(_EXERCISES))
_try("the committee pack", _show_pack, needs=tuple(_EXERCISES))

## 13. What the gate cannot see

The gate proves something narrow and valuable: no figure in the pack is a number the run did
not produce, at the precision it was printed to. It does not prove that a figure sits next to
the right words. A figure traces to a *value*, not to a *meaning*: it traces to the first
result that prints as it, whichever sentence it sits in. And a low-precision figure claims a
band wide enough that some result often falls inside it by chance. The cell below measures
how often a number nobody computed would trace by coincidence against this pack's findings,
at each precision.

In [ ]:
def _show_blind_spots() -> None:
    values = np.array([v for _, v in computed_values(assemble_findings(""))])
    ints = sum(bool(np.any(np.abs(values - k) <= 0.5)) for k in range(0, 31))
    print(f"of the whole numbers 0 to 30, {ints} trace to some result in this pack's findings")
    for d in (2, 3, 4):
        grid = np.arange(0, 10 ** d + 1) / 10 ** d
        half = 0.5 * 10.0 ** -d
        idx = np.clip(np.searchsorted(np.sort(values), grid), 1, values.size - 1)
        srt = np.sort(values)
        near = np.minimum(np.abs(srt[idx] - grid), np.abs(srt[idx - 1] - grid))
        share = float(np.mean(near <= half * (1 + TRACE_SLACK)))
        print(f"of every {d}-decimal number in [0, 1], {share:.1%} trace by coincidence")
    print("\nThe fewer decimals a figure carries, the wider the band it claims, and the more "
          "results\nfall inside that band by chance. A typed figure hides best at low precision.")


_try("blind spots", _show_blind_spots,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 6"))

## 14. Common mistakes

- **Re-raising a recurring finding with today's date.** Its age resets to zero every cycle,
  and a finding that has been open for a year never reaches the committee's escalation list.
- **Reading the clock.** `date.today()` in a tracker gives a pack that cannot be regenerated:
  the same evidence prints different ages on different days. Every date comes from `as_of`.
- **Closing a finding because it was not raised again.** Absence from one cycle's evidence
  is not remediation. The register carries it forward unchanged.
- **Writing the summary above the findings.** A typed opinion survives a critical finding it
  never mentions. An assembled one cannot.
- **Counting NOT SIGNIFICANT as a limitation.** That sample could have seen a difference and
  did not; it is a result. NOT MEASURABLE is the limitation.
- **A tolerance you chose.** A figure claims half a unit in its last place and not a digit
  more; a fixed tolerance is too loose for four decimals and too tight for none.
- **Exempting a class of numbers to quieten the gate.** Integers, percentages, tables and
  headings are exactly where a typed number goes when the others are checked.
- **Harvesting digits from strings.** The typed commentary is a string in the findings; a
  gate that reads it traces the typed number to itself.
- **Writing `12-month` in generated text.** Under this definition a number joined to a word
  by a hyphen is part of a name, and the gate will not check it. Generate "12 months".

The first one is worth seeing rather than believing. The cell below re-raises last quarter's
findings as if they were new, and ages both registers.

In [ ]:
def _show_age_reset() -> None:
    honest = build_register(RAISED, HISTORY, PACK_POLICY, AS_OF)
    amnesiac = build_register(RAISED, (), PACK_POLICY, AS_OF)
    for name, reg in (("carried from the last pack", honest), ("re-raised as new", amnesiac)):
        rows = age_register(reg, AS_OF, PACK_POLICY)
        top = PACK_POLICY["escalation"][-1][1]
        print(f"{name:<27} oldest live finding {max(r.age_days for r in rows):>3} days · "
              f"overdue {sum(r.days_overdue > 0 for r in rows)} · escalated to the {top} "
              f"{sum(r.escalation == top for r in rows)}")


_try("age reset", _show_age_reset, needs=("exercise 1", "exercise 2"))

## 15. Self-check

1. The pack says "PSI 0.25" in the calibration paragraph, and the gate traces it to the PSI
   threshold. What has the gate established?
   - (a) that some computed result prints as 0.25 — not that it is the result the sentence
         claims
   - (b) that the calibration paragraph is correct
   - (c) nothing, because thresholds are policy inputs and should be excluded

2. A colleague widens the gate's tolerance to a fixed 0.0001, and the draft's typed figure
   now traces. The right conclusion is:
   - (a) the typed figure was close enough to be acceptable
   - (b) the tolerance should be 0.0001 for every figure in the pack
   - (c) a figure printed to four decimals now traces to anything within two of its own
         half-units, so a figure copied one digit wrong in its last place can trace too

3. A finding closed in the last pack is raised again this cycle. Recording it as a new finding
   with today's raised date:
   - (a) is correct, because it is a new occurrence
   - (b) resets its age, so ageing and escalation understate how long the problem has lasted
   - (c) is harmless as long as the due date is recomputed

4. A figure in the commentary is arithmetically right but was worked out by hand, and the pack
   refuses to render. The right fix is:
   - (a) register the calculation in the pipeline, so the figure is a computed result, and
         format it from the findings
   - (b) widen the tolerance until it traces
   - (c) move it into a table, where the gate is less strict

5. A comparison is NOT MEASURABLE and another is NOT SIGNIFICANT. Which belongs in the
   limitations section?
   - (a) both, because neither found a difference
   - (b) neither, because neither is a finding
   - (c) only NOT MEASURABLE: that sample could not have detected a material difference, while
         the other could have and did not

Answers are in this lesson's worked solution in the course repository.

In [ ]:
print(f"\nlesson wall time: {time.perf_counter() - _LESSON_T0:.1f}s")

## What you built, and where it goes next

Eight functions and a document that proves itself: a register whose identifiers and raised
dates survive from pack to pack, a tracker that ages and escalates from an explicit date, a
summary whose opinion follows from the findings by rule, a limitations section built from what
the suite could not measure, a definition of a figure precise enough to grade, and a gate that
traces every figure in the pack to a computed result at the precision it was printed to — and
a pack that will not render until it does. That is the last module of this programme: every
earlier module produced evidence, and this one makes sure the committee reads only evidence.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<11} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_register),
                              ("exercise 2", _check_ageing),
                              ("exercise 3", _check_summary),
                              ("exercise 4", _check_limitations),
                              ("exercise 5", _check_figures),
                              ("exercise 6", _check_values),
                              ("exercise 7", _check_gate),
                              ("exercise 8", _check_pack)):
            _try(_name, _check)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))